## Problem: Analyzing AdK Equilibrium Data from `MDAnalysisData`

**Scenario:** You are analyzing the equilibrium dynamics of the Adenylate Kinase (AdK) enzyme. Its trajectory data is available via the `MDAnalysisData` library. You need to fetch this data, load it using `MDAnalysis`, calculate the density of atoms in the simulation box over time, and visualize its fluctuations.

**Task:**
1.  **Install Libraries:** Install `MDAnalysis` and `MDAnalysisData`.
2.  **Fetch Data:** Use `MDAnalysisData` to download the `adk_equilibrium` dataset.
3.  **Load Data:** Load the downloaded trajectory into an `MDAnalysis.Universe` object.
4.  **Modify Data (Calculate Density):** Implement a function to calculate the density for each frame of the simulation. Use the box dimensions provided in the trajectory.
5.  **Plot Data:** Develop a function to visualize the calculated number density over time using Matplotlib.


### Step 1: Install `MDAnalysis` and `MDAnalysisData`

We need to install these libraries to fetch and analyze the molecular dynamics data.

In [ ]:
!pip install MDAnalysis MDAnalysisData

import MDAnalysis as mda
import MDAnalysisData as mda_data
import numpy as np
import matplotlib.pyplot as plt
import os

### Step 2: Fetch AdK Equilibrium Data

We'll use `mda_data.datasets.fetch_adk_equilibrium()` to download the dataset. This function returns a dictionary containing the paths to the topology (PDB) and trajectory (DCD) files.

In [ ]:
# Fetch the AdK equilibrium dataset
data = mda_data.datasets.fetch_adk_equilibrium()

# The 'data' object contains paths to the fetched files
topology_file = data['topology']  # PDB file
trajectory_file = data['trajectory'] # DCD file

print(f"Topology file: {topology_file}")
print(f"Trajectory file: {trajectory_file}")

### Step 3: Load Data using `MDAnalysis`

We will create an `MDAnalysis.Universe` object by loading the topology and trajectory files. This object allows us to access all atoms, residues, and frames of the simulation.

In [ ]:
# Load the data into an MDAnalysis Universe object
universe = mda.Universe(topology_file, trajectory_file)

print(f"Number of atoms in the universe: {universe.select_atoms('all').n_atoms}")
print(f"Number of frames in the trajectory: {universe.trajectory.n_frames}")

### Step 4: Modify Data (Calculate Density)

We will iterate through each frame of the `Universe` object. For each frame, we'll get the simulation box dimensions and calculate the volume. Then, we'll calculate the density as the total mass of the atoms divided by the box volume.

In [ ]:
def calculate_density_mdanalysis(universe):
    """
    Calculates the density for each frame in an MDAnalysis Universe.
    Mass density is total mass (Daltons) divided by volume (Å³).
    """
    total_mass = universe.select_atoms('all').masses.sum()
    times = []
    densities = []

    for ts in universe.trajectory:
        lx, ly, lz, _, _, _ = ts.dimensions
        volume = lx * ly * lz

        if volume > 0: # Avoid division by zero if volume is somehow 0
            density = total_mass / volume
            times.append(ts.time)
            densities.append(density)
        else:
            print(f"Warning: Volume is zero or negative at time {ts.time}. Skipping frame.")

    print("Density calculated for all frames.")
    return np.array(times), np.array(densities)

# Calculate mass density using the loaded MDAnalysis Universe
time_points, densities = calculate_density_mdanalysis(universe)

### Step 5: Plot Density Over Time

Now, we'll plot the calculated density against time to observe its fluctuations throughout the simulation.

In [ ]:
def plot_density_over_time(times, densities, title="Density Over Time (AdK Equilibrium)"):
    """
    Plots the density over time.
    Converts density from Da/Å³ to g/cm³.
    """
    # Conversion factor: 1 Da/Å³ = 1.660539 g/cm³
    # 1 Da = 1.660539e-24 g
    # 1 Å = 1e-8 cm => 1 Å³ = 1e-24 cm³
    # So, (1.660539e-24 g) / (1e-24 cm³) = 1.660539 g/cm³
    conversion_factor = 1.660539 # Approximately, to match g/cm³
    densities_g_cm3 = densities * conversion_factor

    plt.figure(figsize=(10, 6))
    plt.plot(times, densities_g_cm3, label='Density', color='blue')
    plt.xlabel('Time (ps)') # MDAnalysis times are typically in picoseconds
    plt.ylabel('Density (g/cm³)') # Updated units
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Density plot generated with units g/cm³.")

# Plot the results
plot_density_over_time(time_points, densities)

### Calculate Average and Standard Deviation of Mass Density

Let's calculate the mean and standard deviation of the mass density over the entire trajectory to quantify its central tendency and fluctuation.

In [ ]:
# Conversion factor from Da/Å³ to g/cm³
conversion_factor = 1.660539

# Convert densities (which are in Da/Å³) to g/cm³
densities_g_cm3 = densities * conversion_factor

# Calculate the average and standard deviation
average_density = np.mean(densities_g_cm3)
std_dev_density = np.std(densities_g_cm3)

print(f"Average Density: {average_density:.6f} g/cm³")
print(f"Standard Deviation of Density: {std_dev_density:.6f} g/cm³")